In [110]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster,cophenet
from sklearn.preprocessing import StandardScaler
from plotly.subplots import make_subplots

### Estandarización y selección de columnas para cluster de productos

In [111]:
#Leyendo el dataset
resumen_productos = pd.read_csv('../data_clean/resumen_productos.csv')
resumen_productos.head(5)

,ID_Producto,Nombre_producto,Categoría,Precio_Unitario,Stock,Unidades_Vendidas,Venta_Total,Transacciones,%_Unidades,%_Venta,Cancelada,Completa,Pendiente
0,6,Asado,Carnicería,28.56,5137,299,8539.44,81,2.863710,8.282421,0,71,10
1,8,Milanesa,Carnicería,16.21,3140,320,5187.20,89,3.064841,5.031076,0,76,13
2,25,Pizza congelada,Congelados,15.45,1640,332,5129.40,86,3.179772,4.975016,1,79,6
3,4,Queso rallado,Lácteos,19.23,2099,259,4980.57,77,2.480605,4.830665,1,67,9
4,3,Queso cremoso,Lácteos,17.23,3167,268,4617.64,84,2.566804,4.478659,0,66,18


In [112]:
resumen_productos['Porc_Canceladas'] = resumen_productos['Cancelada']/resumen_productos['Transacciones'] *100
resumen_productos['Porc_Completadas'] = resumen_productos['Completa']/resumen_productos['Transacciones'] *100
resumen_productos['Porc_Pendientes'] = resumen_productos['Pendiente']/resumen_productos['Transacciones'] *100


resumen_productos['Porc_Unidades'] = resumen_productos['%_Unidades']/100
resumen_productos['Porc_Ventas'] = resumen_productos['%_Venta']/100

In [113]:
resumen_productos = resumen_productos.drop(columns=['Cancelada', 'Completa', 'Pendiente', '%_Unidades', '%_Venta'])

resumen_productos.head(5)

,ID_Producto,Nombre_producto,Categoría,Precio_Unitario,Stock,Unidades_Vendidas,Venta_Total,Transacciones,Porc_Canceladas,Porc_Completadas,Porc_Pendientes,Porc_Unidades,Porc_Ventas
0,6,Asado,Carnicería,28.56,5137,299,8539.44,81,0.000000,87.654321,12.345679,0.028637,0.082824
1,8,Milanesa,Carnicería,16.21,3140,320,5187.20,89,0.000000,85.393258,14.606742,0.030648,0.050311
2,25,Pizza congelada,Congelados,15.45,1640,332,5129.40,86,1.162791,91.860465,6.976744,0.031798,0.049750
3,4,Queso rallado,Lácteos,19.23,2099,259,4980.57,77,1.298701,87.012987,11.688312,0.024806,0.048307
4,3,Queso cremoso,Lácteos,17.23,3167,268,4617.64,84,0.000000,78.571429,21.428571,0.025668,0.044787


In [114]:
#Filtrando las posibles columnas a usar
interest_cols = ['Unidades_Vendidas', 'Venta_Total', 'Transacciones', 'Porc_Unidades', 'Porc_Ventas', 'Precio_Unitario', 'Porc_Canceladas', 'Porc_Completadas', 'Porc_Pendientes', 'Stock']
X_prod = resumen_productos[interest_cols]
X_prod = X_prod.rename(columns={'Categoría': 'Categoria'})
X_prod.head(5)

,Unidades_Vendidas,Venta_Total,Transacciones,Porc_Unidades,Porc_Ventas,Precio_Unitario,Porc_Canceladas,Porc_Completadas,Porc_Pendientes,Stock
0,299,8539.44,81,0.028637,0.082824,28.56,0.000000,87.654321,12.345679,5137
1,320,5187.20,89,0.030648,0.050311,16.21,0.000000,85.393258,14.606742,3140
2,332,5129.40,86,0.031798,0.049750,15.45,1.162791,91.860465,6.976744,1640
3,259,4980.57,77,0.024806,0.048307,19.23,1.298701,87.012987,11.688312,2099
4,268,4617.64,84,0.025668,0.044787,17.23,0.000000,78.571429,21.428571,3167


In [115]:
# Aplicando el StandardScaler a los datos 
scaler = StandardScaler()
X_scaled_values = scaler.fit_transform(X_prod)
X_prod_scaled = pd.DataFrame(X_scaled_values, columns=X_prod.columns)

X_prod_scaled.head(5)

,Unidades_Vendidas,Venta_Total,Transacciones,Porc_Unidades,Porc_Ventas,Precio_Unitario,Porc_Canceladas,Porc_Completadas,Porc_Pendientes,Stock
0,0.589461,3.640095,0.217133,0.589461,3.640095,3.392403,-0.425004,0.766728,-0.708987,1.746380
1,1.100200,1.545681,1.063395,1.100200,1.545681,1.154800,-0.425004,0.279457,-0.209220,0.002711
2,1.392051,1.509568,0.746046,1.392051,1.509568,1.017101,0.986195,1.673173,-1.895691,-1.307005
3,-0.383374,1.416582,-0.205998,-0.383374,1.416582,1.701971,1.151140,0.628517,-0.854286,-0.906232
4,-0.164486,1.189831,0.534481,-0.164486,1.189831,1.339606,-0.425004,-1.190681,1.298620,0.026286


In [116]:
correlation =  X_prod.corr()

fig = px.imshow(correlation, text_auto=".2f")
fig.update_layout(width = 1200, height = 1200)
fig.show()

Como se puede observar hay diversas variables que tienen una alta correlación entre si lo cuál no es adecuado para lograr hacer clustering, debido a ello se utilizarán únicamente las columnas: 

- Porc_Completadas
- Porc_Canceladas
- Precio_Unitario
- Stock
- Unidades_Vendidas

In [117]:
not_used_cols = ['Venta_Total', 'Transacciones', 'Porc_Unidades', 'Porc_Ventas', 'Porc_Pendientes']

X_prod_scaled = X_prod_scaled.drop(columns=not_used_cols)

X_prod_scaled.head(5)

,Unidades_Vendidas,Precio_Unitario,Porc_Canceladas,Porc_Completadas,Stock
0,0.589461,3.392403,-0.425004,0.766728,1.746380
1,1.100200,1.154800,-0.425004,0.279457,0.002711
2,1.392051,1.017101,0.986195,1.673173,-1.307005
3,-0.383374,1.701971,1.151140,0.628517,-0.906232
4,-0.164486,1.339606,-0.425004,-1.190681,0.026286


## Elección de K

In [122]:
# Usando KMeans para determinar el numero de clusters

k_values = [i for i in range(2, 11)]
inertias = []
silhouette_scores_kmeans = []

for k in k_values:
    k_means = KMeans(n_clusters=k, random_state=42, n_init='auto')
    cluster_labels = k_means.fit_predict(X_prod_scaled)
    inertias.append(k_means.inertia_)
    #print(k_means.labels_)
    silhouette_scores_kmeans.append({'silhouette_score': silhouette_score(X_prod_scaled, cluster_labels)})

summary_scores_kmeans = pd.DataFrame(silhouette_scores_kmeans, index=k_values)

summary_scores_kmeans

,silhouette_score
2,0.147170
3,0.145617
4,0.171391
5,0.151716
6,0.176816
7,0.149911
8,0.165675
9,0.145551
10,0.154393


In [127]:
#Gemini created the subplots

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>Método del codo (Inercia)</b>', '<b>Silhouette Scores</b>')
)

fig.add_trace(go.Scatter(
    x=k_values,
    y=inertias,
    mode='lines+markers',
    name='Intercia (WCSS)',
    marker=dict(color='blue', size=10, line=dict(width=1, color='DarkSlateGrey')),
    line=dict(color='blue', width=2)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=summary_scores_kmeans.index,
    y=summary_scores_kmeans['silhouette_score'],
    mode='lines+markers',
    name='Silhouette Score',
    marker=dict(color='green', size=10, line=dict(width=1, color='DarkSlateGrey')),
    line=dict(color='green', width=2)
), row=1, col=2)

fig.update_layout(
    title=dict(
        text='<b>K-Means Cluster Análisis</b>',
        x=0.5,
        font=dict(size=24, color='black')
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=False, # The subplot titles are clear enough
    hovermode='x unified'
)


fig.update_xaxes(
    title_text='Número de clusters (k)',
    dtick=1, # Tick for every k-value
    gridcolor='rgba(200, 200, 200, 0.5)',
    zeroline=False,
    showline=True,
    linewidth=1,
    linecolor='black',
    row=1, col=1 
)
fig.update_yaxes(
    title_text='Inertia (WCSS)',
    gridcolor='rgba(200, 200, 200, 0.5)',
    zeroline=False,
    showline=True,
    linewidth=1,
    linecolor='black',
    row=1, col=1 
)

# 6. Update axes for Subplot 2 (Silhouette)
fig.update_xaxes(
    title_text='Número de clusters (k)',
    dtick=1,
    gridcolor='rgba(200, 200, 200, 0.5)',
    zeroline=False,
    showline=True,
    linewidth=1,
    linecolor='black',
    row=1, col=2
)
fig.update_yaxes(
    title_text='Silhouette Score',
    gridcolor='rgba(200, 200, 200, 0.5)',
    zeroline=False,
    showline=True,
    linewidth=1,
    linecolor='black',
    row=1, col=2
)

# Show the final combined figure
fig.show()

In [ ]:
# Dendogram
